# RD02 — Latent-regime tournament

Three-state Student-t HMM vs causal age-dependent semi-Markov overlay, same-origin observable tree and continuous return-skew controls. **Historical exploratory only**; no AM-T-HAR-Blend density claim.

In [1]:
import os,sys,time,json,subprocess,tempfile,gc
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').is_dir(),'Run this notebook from repository checkout'
sys.path.insert(0,str(ROOT/'src'))
load_dotenv(ROOT/'.env')
from data_engineering.modeling import latent_regime_tournament as m
SYNTHETIC=os.getenv('RD02_SYNTHETIC','0')=='1'
if SYNTHETIC:
    temp=tempfile.TemporaryDirectory(prefix='rd02-')
    OUT=Path(temp.name)/'02'
    cfg=m.Spec(em_iterations=3,bootstrap_reps=25)
else:
    root=os.getenv('DERIBIT_DATA_ROOT')
    if not root:raise RuntimeError('DERIBIT_DATA_ROOT missing from .env or environment')
    STORE=Path(root)/'research'/'cs04a_feature_store'
    OUT=ROOT/'notebooks'/'Regime_Discovery'/'02'
    cfg=m.Spec()
OUT.mkdir(parents=True,exist_ok=True)
print(m.VERSION,'synthetic=',SYNTHETIC,'output=',OUT)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
check=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_latent_regime_tournament.py')],
                  cwd=str(ROOT),env=env,capture_output=True,text=True)
print(check.stdout[-1500:]);print(check.stderr[-500:]);assert check.returncode==0,'RD02 tests failed; stop'


RD02-0.1.0 synthetic= False output= D:\Projects\deribit_market_data\notebooks\Regime_Discovery\02
...........                                                              [100%]
============================== warnings summary ===============================
tests/test_latent_regime_tournament.py::test_hmm_em_and_causal_filter
  D:\Anaconda\envs\deribit-data\Lib\site-packages\sklearn\cluster\_kmeans.py:1424: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
    warnings.warn(

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
11 passed, 1 warning in 9.31s




In [2]:
panels={};excluded={}
if SYNTHETIC:
    sys.path.insert(0,str(ROOT/'tests'))
    from test_latent_regime_tournament import synthetic_panel
    panels={'BTC':synthetic_panel(days=80,seed=cfg.seed)}
else:
    required=list(dict.fromkeys(['state_time','return_5m','feature_ready','log_rv24h',
                           *m.STATE_FEATURES,*m.RETURN_FEATURES,'ret_5m','ret_15m','ret_30m']))
    for asset in cfg.assets:
        path=STORE/f'{asset}_features_and_forward_magnitude.parquet'
        if not path.is_file():excluded[asset]=f'Missing: {path}';continue
        try:
            panels[asset]=pd.read_parquet(path,columns=required)
            print(asset,len(panels[asset]),'rows loaded')
        except Exception as exc:excluded[asset]=f'{type(exc).__name__}: {exc}'
    if not {'BTC','ETH'}.issubset(panels):
        raise RuntimeError(f'Primary assets missing: {excluded}')
assert panels,excluded


BTC 221761 rows loaded
ETH 221761 rows loaded
SOL 221761 rows loaded
XRP 221761 rows loaded


In [3]:
results={};coverage=[]
for asset,panel in panels.items():
    start=time.perf_counter()
    result=m.scan_asset(panel,asset,cfg)
    results[asset]=result
    coverage.append({'asset':asset,'complete_days':result['complete_days'],
                     'asset_fold_horizon_scores':len(result['scores']),
                     'observable_tree_definitions':len(result['definitions']),
                     'elapsed_seconds':round(time.perf_counter()-start,1)})
    print(asset,'finished in',round(time.perf_counter()-start,1),'s')
    del panel;gc.collect()
coverage=pd.DataFrame(coverage)
display(coverage)


BTC finished in 132.5 s
ETH finished in 134.0 s
SOL finished in 132.3 s
XRP finished in 133.4 s


,asset,complete_days,asset_fold_horizon_scores,observable_tree_definitions,elapsed_seconds
0,BTC,736,72,3,132.5
1,ETH,736,72,3,134.0
2,SOL,736,72,3,132.3
3,XRP,736,72,3,133.4


In [4]:
out=m.consolidate(results,cfg,synthetic=SYNTHETIC)
out['manifest']['excluded']=excluded
print('Mean NLL improvement against fitted continuous-skew benchmark (negative better)')
display(out['comparison'].sort_values(['horizon_min','delta_nll_vs_continuous_skew']))
print('Primary 30m NLL, paired day-bootstrap (historical exploratory):')
display(out['bootstrap'].query('horizon_min==30').head(50))
print('State diagnostics')
display(out['states'].head(18))
print('30-minute validated return by causally filtered HMM state')
display(out['validation_states'].head(20))
print('Decision:',json.dumps(out['decision'],indent=2))


Mean NLL improvement against fitted continuous-skew benchmark (negative better)


,method,horizon_min,delta_nll_vs_continuous_skew,favorable_asset_folds,n_asset_folds
4,observable_tree_skew,5,0.000836,3,12
8,latent_static,5,0.002604,0,12
16,latent_semi_markov,5,0.002671,0,12
12,latent_hmm,5,0.002674,0,12
0,direct_symmetric,5,0.002900,0,12
5,observable_tree_skew,15,0.000278,3,12
1,direct_symmetric,15,0.001246,2,12
9,latent_static,15,0.001343,0,12
13,latent_hmm,15,0.001386,0,12
17,latent_semi_markov,15,0.001395,0,12


Primary 30m NLL, paired day-bootstrap (historical exploratory):


,asset,fold,horizon_min,model,reference,delta_nll,ci025,ci975,n_days,historical_only
12,BTC,1,30,latent_hmm,direct_continuous_skew,-0.001082,-0.003027,0.000877,47,True
13,BTC,1,30,latent_semi_markov,direct_continuous_skew,-0.001097,-0.002688,0.000614,47,True
14,BTC,1,30,latent_semi_markov,latent_hmm,-0.000014,-0.000105,0.000074,47,True
15,BTC,1,30,latent_hmm,latent_static,0.000126,-0.000118,0.000356,47,True
16,BTC,1,30,latent_hmm,observable_tree_skew,-0.000678,-0.002101,0.000725,47,True
17,BTC,1,30,latent_semi_markov,observable_tree_skew,-0.000692,-0.001809,0.000602,47,True
36,BTC,2,30,latent_hmm,direct_continuous_skew,-0.000614,-0.002653,0.001415,47,True
37,BTC,2,30,latent_semi_markov,direct_continuous_skew,-0.000619,-0.002528,0.001622,47,True
38,BTC,2,30,latent_semi_markov,latent_hmm,-0.000005,-0.000027,0.000023,47,True
39,BTC,2,30,latent_hmm,latent_static,-0.000033,-0.000189,0.000129,47,True


State diagnostics


,asset,fold,state,prior,stay_hmm,viterbi_episodes,emission_mean_standardized,structure_obs,loglike_last
0,BTC,1,0,3.190227e-01,0.887072,290,0.039273,9840,-65232.957951
1,BTC,1,1,6.809769e-01,0.923330,286,-0.175482,9840,-65232.957951
2,BTC,1,2,3.333332e-07,0.995976,8,0.349669,9840,-65232.957951
3,BTC,2,0,3.453687e-01,0.883232,403,0.049127,12576,-83139.194132
4,BTC,2,1,3.333332e-07,0.995797,8,0.401572,12576,-83139.194132
5,BTC,2,2,6.546310e-01,0.919901,397,-0.166216,12576,-83139.194132
6,BTC,3,0,3.333332e-07,0.992474,16,0.403280,15312,-102467.842433
7,BTC,3,1,5.880777e-01,0.918415,486,-0.167289,15312,-102467.842433
8,BTC,3,2,4.119220e-01,0.886801,489,0.050074,15312,-102467.842433
9,ETH,1,0,1.335507e-03,0.886583,292,0.019287,9840,-65834.349540


30-minute validated return by causally filtered HMM state


,asset,fold,state,n_forecast_origins,n_days,occupancy,next30_up_frequency,next30_mean_bps,next30_median_bps
0,BTC,1,0,437,42,0.195479,0.482838,0.489486,-1.773489
1,BTC,1,1,1741,47,0.790337,0.504882,0.000179,0.151154
2,BTC,1,2,31,3,0.014184,0.387097,-7.571294,-7.323411
3,BTC,2,0,1278,46,0.578014,0.503130,-1.407496,0.409654
4,BTC,2,2,931,46,0.421986,0.514501,-0.422510,0.727697
5,BTC,3,1,981,41,0.443706,0.502548,-0.709578,0.076581
6,BTC,3,2,1228,44,0.556294,0.507329,-1.856485,0.589043
7,ETH,1,0,1173,47,0.529699,0.517477,0.123899,1.196987
8,ETH,1,1,1016,47,0.461436,0.529528,2.193762,2.093795
9,ETH,1,2,20,1,0.008865,0.400000,-12.643117,-7.198779


Decision: {
  "status": "HISTORICAL_DISCOVERY_NOT_CONFIRMATION",
  "primary_horizon_min": 30,
  "heuristic_favorable": {
    "direct_symmetric": false,
    "observable_tree_skew": false,
    "latent_static": false,
    "latent_hmm": false,
    "latent_semi_markov": false
  },
  "latent_hmm_competitive": false,
  "latent_duration_competitive": false,
  "historical_exploratory": true,
  "prospective_confirmation": false,
  "synthetic": false,
  "AM_T_HAR_Blend_modified": false,
  "warning": "RD02 semi-Markov is a two-stage duration overlay; not a jointly fitted HSMM. Normalization is trailing RV, NOT AM-T-HAR-Blend. Historical development folds already inspected."
}


In [5]:
mapping={
 '02_scores.csv':'scores','02_comparison_summary.csv':'comparison',
 '02_paired_day_bootstrap.csv':'bootstrap','02_state_diagnostics.csv':'states',
 '02_return_response.csv':'responses','02_transition_matrix.csv':'transitions',
 '02_duration_hazards.csv':'hazards','02_validation_state_behavior.csv':'validation_states',
 '02_observable_tree_definitions.csv':'definitions'
}
for name,key in mapping.items():out[key].to_csv(OUT/name,index=False)
# Aggregate durations only, explicitly distinguish right-censored episodes.
ep=out['episodes'].groupby(['asset','fold','state','censored_at_segment_end'],as_index=False).agg(
   episodes=('duration_blocks','size'),mean_duration_blocks=('duration_blocks','mean'),
   median_duration_blocks=('duration_blocks','median'),p90_duration_blocks=('duration_blocks',lambda x:x.quantile(.9)))
ep.to_csv(OUT/'02_filtered_episode_summary.csv',index=False)
coverage.to_csv(OUT/'02_coverage.csv',index=False)
(OUT/'02_manifest.json').write_text(json.dumps(out['manifest'],indent=2,default=str),encoding='utf-8')
(OUT/'02_decision.json').write_text(json.dumps(out['decision'],indent=2,default=str),encoding='utf-8')
assert all((OUT/name).is_file() for name in [*mapping,'02_filtered_episode_summary.csv',
                                       '02_coverage.csv','02_manifest.json','02_decision.json'])
print('Wrote',len(mapping)+4,'aggregates to',OUT,'synthetic=',SYNTHETIC)
print('Never commit synthetic outputs or origin-level sensitive data.')


Wrote 13 aggregates to D:\Projects\deribit_market_data\notebooks\Regime_Discovery\02 synthetic= False
Never commit synthetic outputs or origin-level sensitive data.


## Interpretation limits

The semi-Markov extension is a **two-stage fitted hazard overlay**, not joint HSMM inference. Latent IDs are not stable labels between folds. All forecasts condition on current known features; Viterbi is used only on train for duration fit. Model uses trailing variance proxy, not AM-T-HAR-Blend. A negative 30m NLL delta on studied history is not independent evidence.